# Human and LLM judgment comparison

Configure one or more human aggregate CSVs and an optional LLM log below. Cohort names and row counts are unrestricted. Scores and approval percentages use thresholds of 50 and 60. Stored outputs reflect earlier runs; rerun locally to report on new inputs.


In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file() and (p / 'human_validation').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Set ROOT to the repository directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from human_validation.events.human_llm_statistics import run
from IPython.display import Image, Markdown, display

# Replace or extend this mapping with any aggregate CSVs, regardless of size/name.
# Required: original_row_id, event, human_score, human_approval_pct.
HUMAN_CSVS = {
    'historical': ROOT / 'human_validation/events/data/global_comparison_results.csv',
    'unified': ROOT / 'human_validation/events/data/unified_human_judgments.csv',
}
JUDGE_CSV = ROOT / 'experiments/events/results/tribunal_judgments.csv'
DATASET_CSV = ROOT / 'experiments/events/results/forensic_atomic_tribunal_validated.csv'  # optional category lookup; None to skip
OUTPUT_DIR = ROOT / 'human_validation/events/results/comparison_run'
LLM_SOURCE = 'auto'  # auto / log / embedded; None JUDGE_CSV is allowed with embedded decisions
COHORT_RESULTS = {name: OUTPUT_DIR / name for name in HUMAN_CSVS}

# Optional dataset export, independent of the comparison reports. Set all three
# paths to enable it; leave FILTER_OUTPUT_PATH=None for comparison-only runs.
FILTER_DATASET_PATH = None
FILTER_HUMAN_CSV = None
FILTER_OUTPUT_PATH = None
HUMAN_SCORE_THRESHOLD = 50
HUMAN_APPROVAL_THRESHOLD = 50
FILTER_KEY_COLUMNS = ['event', 'crime_category', 'crime_subcategory', 'brief_context']
FILTER_FALLBACK_COLUMNS = ['event', 'crime_category', 'crime_subcategory']  # [] disables fallback


## LLM and cohort overview


In [ ]:
tables = run(ROOT, human_csvs=HUMAN_CSVS, judge_csv=JUDGE_CSV,
             dataset_csv=DATASET_CSV, output_dir=OUTPUT_DIR, llm_source=LLM_SOURCE)
for name in ('llm_overview', 'llm_votes', 'llm_policy_50_60', 'llm_provider_statistics'):
    if name in tables:
        display(Markdown(f'### {name.replace("_", " ").title()}'), tables[name])
for cohort in HUMAN_CSVS:
    display(Markdown(f'### {cohort} cohort'), tables[f'{cohort}/summary'])


## Threshold comparisons

Score, approval percentage, and score with at least 50% human approval are separate decision rules. TP, TN, FP and FN use the human rule as the reference. Agreement and Cohen kappa describe concordance.


In [ ]:
for cohort in HUMAN_CSVS:
    display(Markdown(f'### {cohort.title()} threshold decisions'), tables[f'{cohort}/comparison_50_60'])


## Approval rates

These charts follow the threshold tables.


In [ ]:
for cohort in HUMAN_CSVS:
    path = COHORT_RESULTS[cohort] / 'approval_50_60.png'
    display(Markdown(f'### {cohort.title()} approval rates'), Image(filename=str(path), width=900))


## Confusion matrices

Rows show the human decision; columns show the official LLM decision.


In [ ]:
for cohort in HUMAN_CSVS:
    display(Markdown(f'### {cohort.title()} confusion matrices'))
    for measure in ('human_score', 'human_approval_pct', 'human_score_and_approval_pct_ge_50'):
        for threshold in (50, 60):
            path = COHORT_RESULTS[cohort] / 'figures' / f'confusion_{measure}_{threshold}.png'
            display(Image(filename=str(path), width=500))


## Form level results and join checks

Per-form comparisons use optional individual responses when available, otherwise `form_id`. Category reports require `crime_category` or the optional category lookup. Missing optional metadata does not prevent the main comparison.


In [ ]:
for cohort in HUMAN_CSVS:
    for report in ('match_audit', 'by_form_50_60', 'by_category_50_60'):
        name = f'{cohort}/{report}'
        if name in tables:
            display(Markdown(f'### {cohort} {report.replace("_", " ")}'), tables[name])


## Human and LLM scores

Score comparisons follow the form and category results.


In [ ]:
for cohort in HUMAN_CSVS:
    path = COHORT_RESULTS[cohort] / 'score_comparison.png'
    display(Markdown(f'### {cohort.title()} score comparison'), Image(filename=str(path), width=600))


## Output and optional dataset filtering

Tables and figures are saved in `OUTPUT_DIR/<cohort>/`. Filtering a dataset is optional: configure the three `FILTER_*_PATH` / `FILTER_HUMAN_CSV` paths and matching columns in the first cell. Only matched scenarios explicitly rejected by the joint score/approval rule are removed. The source dataset is preserved.


In [ ]:
import pandas as pd

if FILTER_OUTPUT_PATH is None:
    print('Dataset filtering disabled. Configure FILTER_* paths to enable it.')
else:
    if FILTER_DATASET_PATH is None or FILTER_HUMAN_CSV is None:
        raise ValueError('Configure FILTER_DATASET_PATH and FILTER_HUMAN_CSV')
    DATASET_PATH = ROOT / Path(FILTER_DATASET_PATH)
    HUMAN_CSV = ROOT / Path(FILTER_HUMAN_CSV)
    OUTPUT_PATH = ROOT / Path(FILTER_OUTPUT_PATH)
    if OUTPUT_PATH.resolve() in {DATASET_PATH.resolve(), HUMAN_CSV.resolve()}:
        raise ValueError('Choose a separate output path to preserve source CSVs')
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    dataset = pd.read_csv(DATASET_PATH, dtype={"original_row_id": "string"}, low_memory=False)
    human = pd.read_csv(HUMAN_CSV, dtype={"original_row_id": "string"}, low_memory=False)

    # Human approval rule
    scores = pd.to_numeric(human["human_score"], errors="coerce")
    approval_pct = pd.to_numeric(
        human["human_approval_pct"], errors="coerce"
    )

    if scores.isna().any() or approval_pct.isna().any():
        raise ValueError("Some human evaluations have missing scores or approval percentages.")

    human["human_approved"] = (
        (scores >= HUMAN_SCORE_THRESHOLD)
        & (approval_pct >= HUMAN_APPROVAL_THRESHOLD)
    )

    # Match using the explicitly configured identity columns
    KEY_COLUMNS = FILTER_KEY_COLUMNS
    if not KEY_COLUMNS:
        raise ValueError("Configure at least one FILTER_KEY_COLUMNS column")
    for name, frame in (("dataset", dataset), ("human CSV", human)):
        missing = set(KEY_COLUMNS + FILTER_FALLBACK_COLUMNS) - set(frame.columns)
        if missing:
            raise ValueError(f"{name}: missing configured matching columns {sorted(missing)}")

    def scenario_keys(frame):
        parts = [
            frame[col].fillna("").astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .str.casefold()
            for col in KEY_COLUMNS
        ]
        return pd.Series(list(zip(*parts)), index=frame.index)

    dataset["_key"] = scenario_keys(dataset)
    human["_key"] = scenario_keys(human)

    # ROBUST HUMAN-TO-DATASET MATCHING

    TRIPLE_COLUMNS = FILTER_FALLBACK_COLUMNS

    def triple_keys(frame):
        parts = [
            frame[col].fillna("").astype(str)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .str.casefold()
            for col in TRIPLE_COLUMNS
        ]
        return pd.Series(list(zip(*parts)), index=frame.index)

    unique_triples = {}
    if TRIPLE_COLUMNS:
        dataset["_key3"] = triple_keys(dataset)
        human["_key3"] = triple_keys(human)
        for triple, group in dataset.groupby("_key3"):
            candidate_keys = group["_key"].drop_duplicates()
            if len(candidate_keys) == 1:
                unique_triples[triple] = candidate_keys.iloc[0]

    known_keys = set(dataset["_key"])

    human["_matched_key"] = human["_key"].where(
        human["_key"].isin(known_keys)
    )

    # Optional fallback, accepted only when it identifies one full key
    need_fallback = human["_matched_key"].isna()

    if TRIPLE_COLUMNS:
        human.loc[need_fallback, "_matched_key"] = (
            human.loc[need_fallback, "_key3"].map(unique_triples)
        )

    # Audit unresolved matches
    unmatched = human["_matched_key"].isna()

    if unmatched.any():
        audit_columns = list(dict.fromkeys(KEY_COLUMNS + TRIPLE_COLUMNS))

        if "original_row_id" in human.columns and "original_row_id" not in audit_columns:
            audit_columns.insert(0, "original_row_id")

        audit = human.loc[unmatched, audit_columns].copy()

        audit_path = OUTPUT_PATH.with_name(
            f"{OUTPUT_PATH.stem}_unmatched_audit.csv"
        )

        audit.to_csv(audit_path, index=False, encoding="utf-8-sig")

        print(f"WARNING: {unmatched.sum()} scenarios remain unmatched.")
        print(f"Audit saved to: {audit_path}")

        display(audit)

    # Only matched scenarios are used to remove dataset rows
    matched_human = human.loc[~unmatched].copy()

    # Check conflicting decisions after matching
    conflicts = (
        matched_human.groupby("_matched_key")["human_approved"]
        .nunique()
    )

    if conflicts.gt(1).any():
        raise ValueError(
            "Conflicting human decisions for the same matched scenario."
        )

    # Build decision mapping
    decisions = (
        matched_human.groupby("_matched_key")["human_approved"]
        .first()
        .to_dict()
    )

    dataset["_human_approved"] = dataset["_key"].map(decisions)

    print("MATCHING RESULTS")
    print("Human evaluations:", len(human))
    print("Matched evaluations:", len(matched_human))
    print("Unmatched evaluations:", int(unmatched.sum()))
    print("Exact matches:", int((~need_fallback).sum()))
    print(
        "Recovered through fallback:",
        int((need_fallback & ~unmatched).sum())
    )

    # Remove only explicitly rejected scenarios
    removed = dataset["_human_approved"].eq(False)

    final_dataset = dataset.loc[~removed].drop(
        columns=["_key", "_key3", "_human_approved"], errors="ignore"
    ).copy()

    # Export without modifying the original dataset
    final_dataset.to_csv(
        OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("FINAL DATASET")
    print(f"Original instances: {len(dataset)}")
    print(f"Human-evaluated scenarios: {len(human)}")
    print(f"Human-approved scenarios: {human['human_approved'].sum()}")
    print(f"Human-rejected scenarios: {(~human['human_approved']).sum()}")
    print(f"Removed dataset rows: {removed.sum()}")
    print(f"Final dataset instances: {len(final_dataset)}")
    print(f"Saved to: {OUTPUT_PATH}")

    display(final_dataset.head())